## Домашнее задание 4. Версия 3 с улучшениями.

Большая часть файлов были расчитаны в предыдущей версии блокнота. 
Датасет с файлами по ссылке: https://kaggle.com/datasets/b772dd8cacfcc95c5d933df56371c4e9695ccdb68ab596aea614c6c58b2ca797


ФИО: Никитченко Мария Владиславовна

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import gc, sys, pickle, os, time
from pathlib import Path
from scipy.sparse import csr_matrix, load_npz as sp_load
from itertools import islice
from collections import Counter, defaultdict

ART_DIR = Path("/kaggle/input/datasets/marusyan132/cashed")
DATA_DIR = Path("/kaggle/input/competitions/hse-26-rec-sys-course-competition")
WORK_DIR = Path('/kaggle/working/')

def trim():
    try:
        import ctypes; ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass

def free(*names):
    g = globals()
    for n in names:
        if n in g: del g[n]
    gc.collect(); trim()

def mem_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e6
    except Exception:
        return -1.0

def save_pickle(obj, name):
    with open(WORK_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, DIR = ART_DIR):
    with open(DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)

print(f"RAM: {mem_mb():.0f} MB")

RAM: 184 MB


## 1. Загрузка артефактов

In [ ]:
# Metadata
items = pd.read_parquet(DATA_DIR / "items.pq")
test_users = pd.read_csv(DATA_DIR / "test_users.csv")
test_user_ids = test_users['user_id'].values
print(f"items: {items.shape}, test_users: {len(test_users):,}")

# Indexes and metadata from artifacts
user2idx_full = load_pickle("user2idx_full")
item2idx_full = load_pickle("item2idx_full")
idx2item_full = load_pickle("idx2item_full")
already_bought_full = load_pickle("already_bought_full")
popular_items_full = load_pickle("popular_items_full")
toppop_rank_full = load_pickle("toppop_rank_full")
print(f"users_full: {len(user2idx_full):,}, items_full: {len(item2idx_full):,}")

# Catalog lookups
item_authors_d = load_pickle("item_authors_d")
item_series_d = load_pickle("item_series_d")
item_cats_d = load_pickle("item_cats_d")

# Content affinity structures
user_top_authors = load_pickle("user_top_authors")
user_top_series = load_pickle("user_top_series")
user_top_categories = load_pickle("user_top_categories")
author2items = load_pickle("author2items")
series2items = load_pickle("series2items")
category2items = load_pickle("category2items")

user_top_authors_full = load_pickle("user_top_authors_full")
user_top_series_full = load_pickle("user_top_series_full")
user_top_categories_full = load_pickle("user_top_categories_full")
author2items_full = load_pickle("author2items_full")
series2items_full = load_pickle("series2items_full")
category2items_full = load_pickle("category2items_full")

print(f"RAM: {mem_mb():.0f} MB")

items: (34323, 4), test_users: 185,282
users_full: 303,549, items_full: 29,117
RAM: 1914 MB


In [ ]:
# User/item features (val + full)
user_features = pd.read_parquet(ART_DIR / "user_features.pq")
item_features = pd.read_parquet(ART_DIR / "item_features.pq")
uf_full = pd.read_parquet(ART_DIR / "UF_FULL_CACHE")   # имя файла случайно с переменной
if_full = pd.read_parquet(ART_DIR / "IF_FULL_CACHE")

print(f"user_features (val): {user_features.shape}")
print(f"item_features (val): {item_features.shape}")
print(f"uf_full: {uf_full.shape}")
print(f"if_full: {if_full.shape}")
print(f"RAM: {mem_mb():.0f} MB")

user_features (val): (343937, 15)
item_features (val): (30995, 21)
uf_full: (348111, 15)
if_full: (31221, 21)
RAM: 2040 MB


## 2. Загрузка raw data (train.pq)

In [12]:
train = pd.read_parquet(DATA_DIR / "train.pq")
train['timestamp'] = pd.to_datetime(train['timestamp'])

In [ ]:
# Temporal split — как в baseline (для val pipeline)
cutoff = train['timestamp'].max() - pd.Timedelta(days=30)
train_fit = train[train['timestamp'] <= cutoff].copy()
val_df = train[train['timestamp'] > cutoff].copy()

val_gt_dict = (
    val_df[val_df['is_purchased']]
    .groupby('user_id')['item_id'].apply(set).to_dict()
)
val_users = list(val_gt_dict.keys())
print(f"train: {train.shape}, train_fit: {train_fit.shape}, val_df: {val_df.shape}")
print(f"val_users: {len(val_users):,}")
print(f"RAM: {mem_mb():.0f} MB")

train: (11373426, 6), train_fit: (11051008, 6), val_df: (322418, 6)
val_users: 39,902
RAM: 17352 MB


In [ ]:
# free('train', 'val_df', 'train_fit')

## 3. Recency features

Для каждого юзера вычисляем:
- **last_N_items**: последние N (=5) купленных айтемов (по timestamp)
- **last_N_authors / series / categories**: соответствующий контент-набор
- **days_since_last_purchase**, **days_since_last_interaction**
- **last_impression_item_set**: items в последнем impression-слейте

Для val pipeline используем `train_fit`, для test — `train`.

In [ ]:
LAST_N = 5  # последние N покупок

def compute_recency_per_user(df, last_n=LAST_N):
    """Возвращает dict с recency-структурами по юзерам.
    df должен иметь 'user_id', 'item_id', 'timestamp', 'is_purchased'.
    """
    df_sorted = df.sort_values(['user_id','timestamp'])
    purchases = df_sorted[df_sorted['is_purchased']]

    # последние N покупок (item_ids)
    last_items = (
        purchases.groupby('user_id')['item_id']
        .agg(lambda s: list(s.tail(last_n)))
        .to_dict()
    )

    # последние N interaction любых типов (item_ids)
    last_inter = (
        df_sorted.groupby('user_id')['item_id']
        .agg(lambda s: list(s.tail(last_n)))
        .to_dict()
    )

    # timestamps
    ref = df['timestamp'].max()
    last_purchase_ts = purchases.groupby('user_id')['timestamp'].max().to_dict()
    last_inter_ts = df_sorted.groupby('user_id')['timestamp'].max().to_dict()
    return {
        'last_items': last_items,
        'last_inter': last_inter,
        'last_purchase_ts': last_purchase_ts,
        'last_inter_ts': last_inter_ts,
        'ref_ts': ref,
    }

# Val
print("Computing recency for val (train_fit)...")
rec_val = compute_recency_per_user(train_fit)
print(f"  users with last_items: {len(rec_val['last_items']):,}")

# Full
print("Computing recency for full (train)...")
rec_full = compute_recency_per_user(train)
print(f"  users with last_items: {len(rec_full['last_items']):,}")

# Кэшируем
save_pickle(rec_val,  "recency_val")
save_pickle(rec_full, "recency_full")
print(f"RAM: {mem_mb():.0f} MB")

Computing recency for val (train_fit)...
  users with last_items: 299,581
Computing recency for full (train)...
  users with last_items: 303,549
RAM: 9982 MB


In [ ]:
# Build recency sets: authors/series/categories per user
def build_recency_sets(last_items_d):
    """Для каждого юзера: множество авторов / серий / категорий
    последних купленных айтемов."""
    a_set, s_set, c_set = {}, {}, {}
    for uid, items_list in last_items_d.items():
        a, s, c = set(), set(), set()
        for it in items_list:
            for x in item_authors_d.get(it, []): a.add(x)
            for x in item_series_d.get(it,  []): s.add(x)
            for x in item_cats_d.get(it,    []): c.add(x)
        a_set[uid] = a; s_set[uid] = s; c_set[uid] = c
    return a_set, s_set, c_set

last_authors_val, last_series_val, last_cats_val = build_recency_sets(rec_val['last_items'])
last_authors_full, last_series_full, last_cats_full = build_recency_sets(rec_full['last_items'])

save_pickle(last_authors_val, "last_authors_val")
save_pickle(last_series_val, "last_series_val")
save_pickle(last_cats_val, "last_cats_val")
save_pickle(last_authors_full,"last_authors_full")
save_pickle(last_series_full, "last_series_full")
save_pickle(last_cats_full, "last_cats_full")
print("Recency sets saved.")

Recency sets saved.


## 4. Item-to-item retrieval из B_ease

Для каждого юзера: последние 5 покупок → для каждой берём top-K похожих айтемов по B_ease →
объединяем, сортируем по сумме score → top-K_RECENT кандидатов.

Используем уже обученные B_ease (val) и B_ease_full (test).

Результат сохраняем в `recent_cands_val.pq` и `recent_cands_test.pq`.

In [ ]:
purchases_fit = train_fit[train_fit['is_purchased']].copy()

all_users_fit = purchases_fit['user_id'].unique()
all_items_fit = purchases_fit['item_id'].unique()

user2idx = {u: i for i, u in enumerate(all_users_fit)}
item2idx = {it: i for i, it in enumerate(all_items_fit)}
idx2item = np.array(all_items_fit)

n_users = len(user2idx)
n_items = len(item2idx)

rows = purchases_fit['user_id'].map(user2idx).values
cols = purchases_fit['item_id'].map(item2idx).values

user_item_csr = csr_matrix(
    (np.ones(len(purchases_fit), dtype=np.float32), (rows, cols)),
    shape=(n_users, n_items)
)
item_user_csr = user_item_csr.T.tocsr()

already_bought: dict[int, set] = (
    purchases_fit.groupby('user_id')['item_id'].apply(set).to_dict()
)

In [ ]:
B_ease = np.load(WORK_DIR / "B_ease.npz")["arr"]
print(f"B_ease: {B_ease.shape}, {B_ease.nbytes/1e9:.2f} GB")
print(f"RAM: {mem_mb():.0f} MB")

save_pickle(user2idx, "user2idx")
save_pickle(item2idx, "item2idx")
save_pickle(idx2item, "idx2item")

B_ease: (28884, 28884), 3.34 GB
RAM: 15300 MB


In [ ]:
# Item-to-item retrieval (val)
K_I2I_PER_ITEM = 50   # сколько похожих брать от каждого последнего айтема
K_I2I_TOTAL = 30   # сколько финально вернуть кандидатов

def i2i_recent_candidates(user_last_items, bought, B, item2idx, idx2item,
                          k_per_item=K_I2I_PER_ITEM, k_total=K_I2I_TOTAL):
    """Для юзера: top-K похожих айтемов на каждый последний → суммируем scores → top-N."""
    score_acc = {}     # candidate_item → cumulative B-score
    for it in user_last_items:
        if it not in item2idx: continue
        idx = item2idx[it]
        row = B[idx]  # 1 × n_items
        # top-K per item (быстро через argpartition)
        if len(row) > k_per_item:
            top_k_idx = np.argpartition(row, -k_per_item)[-k_per_item:]
        else:
            top_k_idx = np.arange(len(row))
        for j in top_k_idx:
            cand_item = int(idx2item[j])
            if cand_item in bought or cand_item == it:
                continue
            score_acc[cand_item] = score_acc.get(cand_item, 0.0) + float(row[j])
    if not score_acc: return []
    # сортируем + top-K_TOTAL
    sorted_cands = sorted(score_acc.items(), key=lambda kv: -kv[1])[:k_total]
    return sorted_cands  # list of (item_id, score)

# Already purchased for val users (from train_fit)
purchases_fit_set = (
    train_fit[train_fit['is_purchased']]
    .groupby('user_id')['item_id'].apply(set).to_dict()
)

# Генерим
print("Generating i2i recent candidates for val...")
t0 = time.time()
recent_cands_val = {}
last_items_val = rec_val['last_items']
for uid in val_users:
    last = last_items_val.get(uid, [])
    if not last: continue
    bought = purchases_fit_set.get(uid, set())
    recent_cands_val[uid] = i2i_recent_candidates(last, bought, B_ease, item2idx, idx2item)
print(f"  done in {time.time()-t0:.1f}s, users with i2i cands: {len(recent_cands_val):,}")
print(f"RAM: {mem_mb():.0f} MB")

# Сохраняем как DataFrame
rows_u, rows_i, rows_s, rows_r = [], [], [], []
for uid, lst in recent_cands_val.items():
    for r, (it, sc) in enumerate(lst):
        rows_u.append(uid); rows_i.append(it); rows_s.append(sc); rows_r.append(r)

recent_df_val = pd.DataFrame({
    'user_id': np.array(rows_u, dtype=np.int64),
    'item_id': np.array(rows_i, dtype=np.int64),
    'i2i_score': np.array(rows_s, dtype=np.float32),
    'i2i_rank': np.array(rows_r, dtype=np.int32),
})
recent_df_val.to_parquet(WORK_DIR / "recent_cands_val.pq", index=False)
free('recent_cands_val', 'rows_u','rows_i','rows_s','rows_r','purchases_fit_set','B_ease','user2idx','item2idx','idx2item')
print(f"recent_df_val: {recent_df_val.shape}, RAM: {mem_mb():.0f} MB")

Generating i2i recent candidates for val...
  done in 93.5s, users with i2i cands: 35,934
RAM: 15894 MB


OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/marusyan132/cashed/recent_cands_val.pq'

In [22]:
recent_df_val.to_parquet(WORK_DIR / "recent_cands_val.pq", index=False)
free('recent_cands_val', 'rows_u','rows_i','rows_s','rows_r','purchases_fit_set','B_ease','user2idx','item2idx','idx2item')
print(f"recent_df_val: {recent_df_val.shape}, RAM: {mem_mb():.0f} MB")

recent_df_val: (1078006, 4), RAM: 9252 MB


In [ ]:
B_ease_full = np.load(ART_DIR / "B_ease_full.npz")["arr"]
print(f"B_ease_full: {B_ease_full.shape}, RAM: {mem_mb():.0f} MB")


print("Generating i2i recent candidates for test...")
t0 = time.time()
recent_cands_test = {}
last_items_full = rec_full['last_items']
for uid in test_user_ids:
    last = last_items_full.get(uid, [])
    if not last: continue
    bought = already_bought_full.get(uid, set())
    recent_cands_test[uid] = i2i_recent_candidates(
        last, bought, B_ease_full, item2idx_full, idx2item_full
    )
print(f"  done in {time.time()-t0:.1f}s, users with i2i cands: {len(recent_cands_test):,}")

rows_u, rows_i, rows_s, rows_r = [], [], [], []
for uid, lst in recent_cands_test.items():
    for r, (it, sc) in enumerate(lst):
        rows_u.append(uid); rows_i.append(it); rows_s.append(sc); rows_r.append(r)

recent_df_test = pd.DataFrame({
    'user_id': np.array(rows_u, dtype=np.int64),
    'item_id':  np.array(rows_i, dtype=np.int64),
    'i2i_score': np.array(rows_s, dtype=np.float32),
    'i2i_rank': np.array(rows_r, dtype=np.int32),
})
recent_df_test.to_parquet(WORK_DIR / "recent_cands_test.pq", index=False)
free('recent_cands_test','rows_u','rows_i','rows_s','rows_r','B_ease_full')
print(f"recent_df_test: {recent_df_test.shape}, RAM: {mem_mb():.0f} MB")

B_ease_full: (29117, 29117), RAM: 12645 MB
Generating i2i recent candidates for test...
  done in 419.9s, users with i2i cands: 170,113
recent_df_test: (5103390, 4), RAM: 9375 MB


## 4b. ItemKNN (cosine)

Отдельный retrieval-источник: item-item cosine similarity по purchase-матрице.
Дополняет EASE (тот делает item-item через линейную регрессию) — ItemKNN ловит
более локальные / последовательные паттерны.

Кэшируем `(knn_idx, knn_sim)` массивы формы `(n_items, K)`.

In [27]:
pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 58.0 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [28]:
from implicit.nearest_neighbours import CosineRecommender

In [ ]:
# Build ItemKNN cosine (val + full)
try:
    from implicit.nearest_neighbours import CosineRecommender
    IMP_OK = True
except ImportError:
    IMP_OK = False
    print("Install: pip install implicit")

ITEMKNN_K = 100   # сколько соседей на айтем

def build_itemknn(user_item_csr, K=ITEMKNN_K):
    """Cosine ItemKNN. Возвращает (knn_idx, knn_sim) формы (n_items, K)."""
    model = CosineRecommender(K=K)
    model.fit(user_item_csr)
    sim = model.similarity  # csr (n_items, n_items)
    n_items = sim.shape[0]
    knn_idx = np.full((n_items, K), -1, dtype=np.int32)
    knn_sim = np.zeros((n_items, K), dtype=np.float32)
    for i in range(n_items):
        st, en = sim.indptr[i], sim.indptr[i+1]
        ind = sim.indices[st:en]
        dat = sim.data[st:en]
        if len(ind) == 0: continue
        order = np.argsort(-dat)
        kk = min(len(order), K)
        knn_idx[i, :kk] = ind[order[:kk]]
        knn_sim[i, :kk] = dat[order[:kk]]
    return knn_idx, knn_sim

# val pipeline
KNN_VAL_IDX = WORK_DIR / "itemknn_val_idx.npy"
KNN_VAL_SIM = WORK_DIR / "itemknn_val_sim.npy"

if KNN_VAL_IDX.exists() and KNN_VAL_SIM.exists():
    knn_idx_val = np.load(KNN_VAL_IDX)
    knn_sim_val = np.load(KNN_VAL_SIM)
    print(f"ItemKNN val загружен из кэша: {knn_idx_val.shape}")
else:
    print("Строим user_item_csr (val) из purchases_fit...")
    if 'train_fit' not in globals():
        _train_tmp = pd.read_parquet(DATA_DIR / "train.pq")
        _train_tmp['timestamp'] = pd.to_datetime(_train_tmp['timestamp'])
        cutoff_ = _train_tmp['timestamp'].max() - pd.Timedelta(days=30)
        train_fit = _train_tmp[_train_tmp['timestamp'] <= cutoff_].copy()
        del _train_tmp
    purchases_fit_tmp = train_fit[train_fit['is_purchased']][['user_id','item_id']]
    rows = purchases_fit_tmp['user_id'].map(user2idx).values
    cols = purchases_fit_tmp['item_id'].map(item2idx).values
    user_item_csr_val = csr_matrix(
        (np.ones(len(purchases_fit_tmp), dtype=np.float32), (rows, cols)),
        shape=(len(user2idx), len(item2idx))
    )
    del purchases_fit_tmp, rows, cols
    gc.collect()

    print(f"Обучение ItemKNN cosine на val (n_items={user_item_csr_val.shape[1]})...")
    t0 = time.time()
    knn_idx_val, knn_sim_val = build_itemknn(user_item_csr_val, K=ITEMKNN_K)
    print(f"  done in {time.time()-t0:.1f}s, shape: {knn_idx_val.shape}")
    np.save(KNN_VAL_IDX, knn_idx_val)
    np.save(KNN_VAL_SIM, knn_sim_val)
    del user_item_csr_val
    gc.collect()

# full pipeline
KNN_FULL_IDX = WORK_DIR / "itemknn_full_idx.npy"
KNN_FULL_SIM = WORK_DIR / "itemknn_full_sim.npy"

if KNN_FULL_IDX.exists() and KNN_FULL_SIM.exists():
    knn_idx_full = np.load(KNN_FULL_IDX)
    knn_sim_full = np.load(KNN_FULL_SIM)
    print(f"ItemKNN full загружен из кэша: {knn_idx_full.shape}")
else:
    print("Обучение ItemKNN cosine на full train...")
    user_item_full = sp_load(str(ART_DIR / "user_item_full.npz"))
    t0 = time.time()
    knn_idx_full, knn_sim_full = build_itemknn(user_item_full, K=ITEMKNN_K)
    print(f"  done in {time.time()-t0:.1f}s, shape: {knn_idx_full.shape}")
    np.save(KNN_FULL_IDX, knn_idx_full)
    np.save(KNN_FULL_SIM, knn_sim_full)
    free('user_item_full')

print(f"RAM: {mem_mb():.0f} MB")

Строим user_item_csr (val) из purchases_fit...
Обучение ItemKNN cosine на val (n_items=28884)...


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.03346848487854004 seconds
  warnings.warn(


  0%|          | 0/28884 [00:00<?, ?it/s]

  done in 1.6s, shape: (28884, 100)
Обучение ItemKNN cosine на full train...


/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.03608250617980957 seconds
  warnings.warn(


  0%|          | 0/29117 [00:00<?, ?it/s]

  done in 1.7s, shape: (29117, 100)
RAM: 9379 MB


## 4c. ItemKNN aggregate features

Для каждого юзера:
1. берём последние LAST_N покупок (из `rec_val`/`rec_full`)
2. для каждой смотрим top-K соседей по ItemKNN
3. агрегируем per candidate:
   - `itemknn_score_max` — макс similarity к любому из последних N
   - `itemknn_score_sum` — сумма similarity
   - `itemknn_rank_min` — лучший rank среди соседей
   - `itemknn_n_sources` — сколько из last N айтемов рекомендуют этого кандидата
   - `itemknn_from_last_1/3/5` — max similarity от 1/3/5 последних

Сохраняем как DataFrame: `itemknn_features_val.pq`, `itemknn_features_test.pq`.

In [ ]:
def compute_itemknn_features(last_items_dict, knn_idx, knn_sim, item2idx_, idx2item_,
                              user_ids, last_n=LAST_N):
    """Для каждого юзера: агрегируем ItemKNN-фичи по последним покупкам.
    Возвращает DataFrame с user_id, item_id и 7 фичами."""
    rows = []
    for uid in user_ids:
        last_items = last_items_dict.get(uid, [])
        if not last_items: continue
        # последний айтем = последний в списке (max timestamp)
        last_items_rev = list(reversed(last_items))   # rev[0] = most recent
        last_items_rev = last_items_rev[:last_n]

        agg = {}  # cand_item → [max, sum, rank_min, n_src, from_1, from_3, from_5]
        for pos, item_id in enumerate(last_items_rev):
            if item_id not in item2idx_: continue
            idx = item2idx_[item_id]
            nb_arr = knn_idx[idx]
            sm_arr = knn_sim[idx]
            for rank, (nb_idx, sm_val) in enumerate(zip(nb_arr, sm_arr)):
                if nb_idx == -1 or sm_val <= 0: break
                cand = int(idx2item_[nb_idx])
                if cand == item_id: continue
                a = agg.get(cand)
                if a is None:
                    a = [0.0, 0.0, ITEMKNN_K, 0, 0.0, 0.0, 0.0]
                    agg[cand] = a
                if sm_val > a[0]: a[0] = sm_val
                a[1] += sm_val
                if rank < a[2]: a[2] = rank
                a[3] += 1
                if pos == 0 and sm_val > a[4]: a[4] = sm_val
                if pos < 3  and sm_val > a[5]: a[5] = sm_val
                if pos < 5  and sm_val > a[6]: a[6] = sm_val
        for cand, a in agg.items():
            rows.append((uid, cand, a[0], a[1], a[2], a[3], a[4], a[5], a[6]))

    df = pd.DataFrame(rows, columns=[
        'user_id','item_id',
        'itemknn_score_max','itemknn_score_sum','itemknn_rank_min','itemknn_n_sources',
        'itemknn_from_last_1','itemknn_from_last_3','itemknn_from_last_5',
    ])
    df['user_id'] = df['user_id'].astype(np.int64)
    df['item_id'] = df['item_id'].astype(np.int64)
    df['itemknn_score_max'] = df['itemknn_score_max'].astype(np.float32)
    df['itemknn_score_sum'] = df['itemknn_score_sum'].astype(np.float32)
    df['itemknn_rank_min'] = df['itemknn_rank_min'].astype(np.int32)
    df['itemknn_n_sources'] = df['itemknn_n_sources'].astype(np.int8)
    df['itemknn_from_last_1'] = df['itemknn_from_last_1'].astype(np.float32)
    df['itemknn_from_last_3'] = df['itemknn_from_last_3'].astype(np.float32)
    df['itemknn_from_last_5'] = df['itemknn_from_last_5'].astype(np.float32)
    return df

# val
KNN_FEAT_VAL = WORK_DIR / "itemknn_features_val.pq"
if KNN_FEAT_VAL.exists():
    knn_feat_val = pd.read_parquet(KNN_FEAT_VAL)
    print(f"ItemKNN features val загружены: {knn_feat_val.shape}")
else:
    print("Computing ItemKNN features for val...")
    t0 = time.time()
    # if 'user2idx' not in globals():
        # user2idx = load_pickle("user2idx_val")
        # item2idx = load_pickle("item2idx_val")
        # idx2item = load_pickle("idx2item_val")
    knn_feat_val = compute_itemknn_features(
        rec_val['last_items'], knn_idx_val, knn_sim_val,
        item2idx, idx2item, val_users
    )
    print(f"  done in {time.time()-t0:.1f}s, shape: {knn_feat_val.shape}")
    knn_feat_val.to_parquet(KNN_FEAT_VAL, index=False)

# test
KNN_FEAT_TEST = WORK_DIR / "itemknn_features_test.pq"
if KNN_FEAT_TEST.exists():
    knn_feat_test = pd.read_parquet(KNN_FEAT_TEST)
    print(f"ItemKNN features test загружены: {knn_feat_test.shape}")
else:
    print("Computing ItemKNN features for test...")
    t0 = time.time()
    knn_feat_test = compute_itemknn_features(
        rec_full['last_items'], knn_idx_full, knn_sim_full,
        item2idx_full, idx2item_full, test_user_ids
    )
    print(f"  done in {time.time()-t0:.1f}s, shape: {knn_feat_test.shape}")
    knn_feat_test.to_parquet(KNN_FEAT_TEST, index=False)

print(f"RAM: {mem_mb():.0f} MB")

free('knn_idx_val','knn_sim_val','knn_idx_full','knn_sim_full')

Computing ItemKNN features for val...
  done in 46.7s, shape: (10605194, 9)
Computing ItemKNN features for test...
  done in 220.0s, shape: (43649703, 9)
RAM: 11811 MB


## 5. Augment val candidates (`candidates_val.pq`) новыми фичами

Загружаем существующий `candidates_val.pq` и добавляем:
- `last_author_match`, `last_series_match`, `last_category_match` (recency-aware match)
- `days_since_last_purchase`, `days_since_last_interaction`
- `i2i_score`, `i2i_rank`, `in_i2i_recent` (от item-to-item retrieval)
- `user_activity_bucket` + interaction features
- `was_impressed`, `n_times_impressed` (если есть hard_negatives_fit + purchases_fit)


In [ ]:
# Освобождаем тяжёлое из предыдущих секций (если что-то осталось)
free('train', 'train_fit', 'val_df')

LAST_N = 5
K_I2I_TOTAL = 30      # из секции 4
ITEMKNN_K = 100     # из секции 4b

cand = pd.read_parquet(ART_DIR / "candidates_val.pq")
print(f"cand: {cand.shape}, RAM: {mem_mb():.0f} MB")


item_authors_d = load_pickle("item_authors_d")
item_series_d = load_pickle("item_series_d")
item_cats_d = load_pickle("item_cats_d")

rec_val  = load_pickle("recency_val")
last_authors_val = load_pickle("last_authors_val")
last_series_val = load_pickle("last_series_val")
last_cats_val = load_pickle("last_cats_val")
print(f"recency_val users: {len(rec_val['last_items']):,}")

RECENT_VAL_PATH = ART_DIR / "recent_cands_val.pq"
KNN_VAL_PATH = ART_DIR / "itemknn_features_val.pq"

if RECENT_VAL_PATH.exists():
    recent_df_val = pd.read_parquet(RECENT_VAL_PATH)
    print(f"recent_df_val: {recent_df_val.shape}")
else:
    recent_df_val = pd.DataFrame(columns=['user_id','item_id','i2i_score','i2i_rank'])
    print("recent_cands_val.pq не найден — секция 4 (EASE-i2i) не была запущена.")

if KNN_VAL_PATH.exists():
    knn_feat_val = pd.read_parquet(KNN_VAL_PATH)
    print(f"knn_feat_val: {knn_feat_val.shape}")
else:
    knn_feat_val = pd.DataFrame(columns=[
        'user_id','item_id','itemknn_score_max','itemknn_score_sum',
        'itemknn_rank_min','itemknn_n_sources',
        'itemknn_from_last_1','itemknn_from_last_3','itemknn_from_last_5'
    ])
    print("  ⚠ itemknn_features_val.pq не найден — секции 4b/4c не были запущены.")

print(f"RAM после загрузки: {mem_mb():.0f} MB")

cand: (9258629, 52), RAM: 10763 MB
recency_val users: 299,581
recent_df_val: (1078006, 4)
knn_feat_val: (10605194, 9)
RAM после загрузки: 7100 MB


In [ ]:
# 5a. i2i фичи 
def add_pair_feature(base, src_df, value_col, fill_value, dtype, out_col=None):
    """RAM-friendly merge через MultiIndex.map (пик ~1 GB вместо ~6 GB)."""
    if out_col is None: out_col = value_col
    src_idx = pd.MultiIndex.from_arrays(
        [src_df['user_id'].to_numpy(), src_df['item_id'].to_numpy()],
        names=['user_id','item_id'])
    src_s = pd.Series(src_df[value_col].to_numpy(), index=src_idx)
    base_idx = pd.MultiIndex.from_arrays(
        [base['user_id'].to_numpy(), base['item_id'].to_numpy()],
        names=['user_id','item_id'])
    vals = src_s.reindex(base_idx).to_numpy()
    del src_idx, src_s, base_idx
    vals = np.nan_to_num(vals, nan=fill_value).astype(dtype)
    base[out_col] = vals
    del vals
    gc.collect()

print(f"cand перед 5a: {cand.shape}, RAM: {mem_mb():.0f} MB")

# i2i от EASE
add_pair_feature(cand, recent_df_val, 'i2i_score', 0.0, np.float32)
add_pair_feature(cand, recent_df_val, 'i2i_rank',  K_I2I_TOTAL, np.int32)
cand['in_i2i_recent'] = (cand['i2i_rank'] < K_I2I_TOTAL).astype(np.int8)
free('recent_df_val')
print(f"  i2i features merged. cand: {cand.shape}, RAM: {mem_mb():.0f} MB")

cand перед 5a: (9258629, 52), RAM: 7100 MB
  i2i features merged. cand: (9258629, 55), RAM: 6976 MB


In [ ]:
# 5b. ItemKNN фичи
print("Merging ItemKNN features...")

add_pair_feature(cand, knn_feat_val, 'itemknn_score_max',   0.0, np.float32)
add_pair_feature(cand, knn_feat_val, 'itemknn_score_sum',   0.0,np.float32)
add_pair_feature(cand, knn_feat_val, 'itemknn_rank_min',    ITEMKNN_K, np.int32)
add_pair_feature(cand, knn_feat_val, 'itemknn_n_sources',   0, np.int8)
add_pair_feature(cand, knn_feat_val, 'itemknn_from_last_1', 0.0, np.float32)
add_pair_feature(cand, knn_feat_val, 'itemknn_from_last_3', 0.0, np.float32)
add_pair_feature(cand, knn_feat_val, 'itemknn_from_last_5', 0.0, np.float32)
cand['in_itemknn'] = (cand['itemknn_rank_min'] < ITEMKNN_K).astype(np.int8)
free('knn_feat_val')
print(f"cand после ItemKNN: {cand.shape}, RAM: {mem_mb():.0f} MB")

Merging ItemKNN features...
cand после ItemKNN: (9258629, 63), RAM: 7217 MB


In [ ]:
# 5c. Recency-match фичи
print("Computing recency-match features (chunked)...")
n = len(cand)
lam = np.zeros(n, dtype=np.int8)   # last_author_match
lsm = np.zeros(n, dtype=np.int8)
lcm = np.zeros(n, dtype=np.int8)
dslp = np.full(n, -1, dtype=np.float32)  # days_since_last_purchase
dsli = np.full(n, -1, dtype=np.float32)  # days_since_last_interaction

_iad = item_authors_d; _isd = item_series_d; _icd = item_cats_d
_la = last_authors_val; _ls = last_series_val; _lc = last_cats_val
_lpt = rec_val['last_purchase_ts']; _lit = rec_val['last_inter_ts']
ref_ts = rec_val['ref_ts']

uids = cand['user_id'].values
iids = cand['item_id'].values
CHUNK = 2_000_000

for start in range(0, n, CHUNK):
    end = min(start + CHUNK, n)
    for k in range(start, end):
        uid = int(uids[k]); it = int(iids[k])
        i_a = _iad.get(it, []); i_s = _isd.get(it, []); i_c = _icd.get(it, [])
        a_set = _la.get(uid, ()); s_set = _ls.get(uid, ()); c_set = _lc.get(uid, ())
        lam[k] = 1 if any(a in a_set for a in i_a) else 0
        lsm[k] = 1 if any(s in s_set for s in i_s) else 0
        lcm[k] = 1 if any(c in c_set for c in i_c) else 0
        lp = _lpt.get(uid)
        li = _lit.get(uid)
        if lp is not None: dslp[k] = (ref_ts - lp).days
        if li is not None: dsli[k] = (ref_ts - li).days
    print(f"  {end:,}/{n:,}  RAM={mem_mb():.0f} MB")

cand['last_author_match'] = lam
cand['last_series_match']  = lsm
cand['last_category_match'] = lcm
cand['days_since_last_purchase'] = dslp
cand['days_since_last_interaction'] = dsli
del lam, lsm, lcm, dslp, dsli, uids, iids
free()
print(f"cand: {cand.shape}, RAM: {mem_mb():.0f} MB")

Computing recency-match features (chunked)...
  2,000,000/9,258,629  RAM=7301 MB
  4,000,000/9,258,629  RAM=7305 MB
  6,000,000/9,258,629  RAM=7312 MB
  8,000,000/9,258,629  RAM=7317 MB
  9,258,629/9,258,629  RAM=7319 MB
cand: (9258629, 68), RAM: 7319 MB


In [ ]:
def activity_bucket(n):
    if n <= 0: return 0
    if n <= 5: return 1
    if n <= 20: return 2
    if n <= 50: return 3
    return 4

def _find_act_col(df):
    for c in ['user_n_interactions', 'n_interactions_x']:
        if c in df.columns:
            return c
    return None

#5d. Activity bucket + interaction features
_act = _find_act_col(cand)
if _act is not None:
    cand['user_activity_bucket'] = cand[_act].fillna(0).apply(activity_bucket).astype(np.int8)
    print(f"user_activity_bucket из колонки '{_act}'")
else:
    cand['user_activity_bucket'] = np.int8(0)
    print("⚠ колонка интеракций не найдена — bucket=0")

bucket_f = cand['user_activity_bucket'].astype(np.float32)
for src in ['ease_rank','ials_score','content_rank','toppop_rank','i2i_rank']:
    if src in cand.columns:
        cand[f'{src}_x_bucket'] = (cand[src].astype(np.float32) * bucket_f).astype(np.float32)
del bucket_f
print(f"Activity bucket + interactions added. cand: {cand.shape}")
print(f"  bucket distribution: {cand['user_activity_bucket'].value_counts().to_dict()}")

user_activity_bucket из колонки 'n_interactions_x'
Activity bucket + interactions added. cand: (9258629, 74)
  bucket distribution: {4: 4261597, 3: 2295895, 2: 1843966, 1: 857171}


In [ ]:
# 5e. Impression-based фичи
# was_impressed = item был показан юзеру (in hard_negatives_fit OR в его purchases)
hard_neg_fit = load_pickle("hard_negatives_fit")

was_impressed = np.zeros(len(cand), dtype=np.int8)
uids = cand['user_id'].values
iids = cand['item_id'].values
for k in range(len(cand)):
    uid = int(uids[k]); it = int(iids[k])
    if it in hard_neg_fit.get(uid, set()):
        was_impressed[k] = 1
cand['was_impressed'] = was_impressed

In [10]:

# Сохраняем расширенный cand
cand.to_parquet(WORK_DIR / "candidates_val_v3.pq", index=False)
print(f"Saved candidates_val_v2.pq: {cand.shape}")

free('hard_neg_fit','uids','iids','was_impressed','recent_df_val')
print(f"RAM: {mem_mb():.0f} MB")

Saved candidates_val_v2.pq: (9258629, 75)
RAM: 6725 MB


## 6.LGBM training



In [23]:
cand = pd.read_parquet('/kaggle/working/candidates_val_v3.pq')

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

free('recent_df_val', 'knn_feat_val',
     'last_authors_val', 'last_series_val', 'last_cats_val',
     'item_authors_d', 'item_series_d', 'item_cats_d',
     'rec_val', 'rec_full',
     'last_authors_full', 'last_series_full', 'last_cats_full',
     'B_ease', 'B_ease_full')
print(f"После очистки: cand {cand.shape}, RAM {mem_mb():.0f} MB")

META_COLS = ['user_id','item_id','target','was_impressed']
FEAT_COLS = [c for c in cand.columns if c not in META_COLS]
print(f"Features ({len(FEAT_COLS)})")


for col in FEAT_COLS:
    if cand[col].isna().any():
        cand[col] = cand[col].fillna(-1)
    if cand[col].dtype == 'float64':
        cand[col] = cand[col].astype('float32')
gc.collect(); trim()
print(f"После fillna+downcast: RAM {mem_mb():.0f} MB")


cand = cand.sort_values('user_id').reset_index(drop=True)
gc.collect(); trim()
print(f"После сортировки: RAM {mem_mb():.0f} MB")


X_all = cand[FEAT_COLS].to_numpy(dtype=np.float32, copy=True)
y_all = cand['target'].to_numpy(dtype=np.float32)
user_ids_sorted = cand['user_id'].to_numpy()
item_ids_sorted = cand['item_id'].to_numpy()
print(f"X_all: {X_all.nbytes/1e9:.2f} GB, y_all: {y_all.nbytes/1e9:.2f} GB")


_, all_groups = np.unique(user_ids_sorted, return_counts=True)
print(f"all_groups: {len(all_groups):,} users")


free('cand')
print(f"После free cand: RAM {mem_mb():.0f} MB")


unique_users = np.unique(user_ids_sorted)
train_rr_users, val_rr_users = train_test_split(unique_users, test_size=0.2, random_state=42)
train_set = set(train_rr_users.tolist())
val_set   = set(val_rr_users.tolist())
del unique_users


mask_train = np.fromiter((uid in train_set for uid in user_ids_sorted),
                          dtype=bool, count=len(user_ids_sorted))


X_train = np.ascontiguousarray(X_all[mask_train])
y_train = y_all[mask_train]
X_val   = np.ascontiguousarray(X_all[~mask_train])
y_val   = y_all[~mask_train]
print(f"X_train: {X_train.nbytes/1e9:.2f} GB, X_val: {X_val.nbytes/1e9:.2f} GB")


train_user_ids = user_ids_sorted[mask_train]
val_user_ids   = user_ids_sorted[~mask_train]
val_item_ids   = item_ids_sorted[~mask_train]
_, train_groups = np.unique(train_user_ids, return_counts=True)
_, val_groups   = np.unique(val_user_ids,   return_counts=True)


val_rr = pd.DataFrame({'user_id': val_user_ids, 'item_id': val_item_ids})

После очистки: cand (9258629, 75), RAM 16399 MB
Features (71)
После fillna+downcast: RAM 17229 MB
После сортировки: RAM 19053 MB
X_all: 2.63 GB, y_all: 0.04 GB
all_groups: 36,611 users
После free cand: RAM 17118 MB
X_train: 2.10 GB, X_val: 0.53 GB


In [25]:
# val_gt_rr (только val юзеры)
val_gt_rr = {uid: items for uid, items in val_gt_dict.items() if uid in val_set}

del user_ids_sorted, item_ids_sorted, train_user_ids, val_user_ids, val_item_ids, mask_train
gc.collect(); trim()

print(f"\ntrain: {len(X_train):,} rows / {len(train_groups):,} users")
print(f"val:   {len(X_val):,}   rows / {len(val_groups):,} users")
print(f"RAM итого: {mem_mb():.0f} MB")


train: 7,403,935 rows / 29,288 users
val:   1,854,694   rows / 7,323 users
RAM итого: 19003 MB


In [ ]:
def scores_to_ndcg(scores, df, gt_dict, k=20):
    df = df.copy()
    df['_s'] = scores
    recs = (
        df.sort_values(['user_id','_s'], ascending=[True, False])
        .groupby('user_id', sort=False)
        .head(k)
        .groupby('user_id', sort=False)['item_id']
        .apply(list).to_dict()
    )
    def dcg(rels): return sum(r / np.log2(i+2) for i, r in enumerate(rels))
    out = []
    for uid, items_list in recs.items():
        gt = gt_dict.get(uid, set())
        if not gt: continue
        rels = [1.0 if it in gt else 0.0 for it in items_list]
        idcg = dcg([1.0]*min(len(gt), k))
        if idcg == 0: continue
        out.append(dcg(rels) / idcg)
    return np.mean(out) if out else 0.0


lgbm_params = {
    'n_estimators':      528,
    'num_leaves':        36,
    'learning_rate':     0.18258230439200238,
    'min_child_samples': 40,
    'lambda_l1':         2.9866092370009407,
    'lambda_l2':         2.0414536703077757,
    'subsample':         0.8391599915244341,
    'colsample_bytree':  0.9687496940092467,
    'objective':         'lambdarank',
    'metric':            'ndcg',
    'ndcg_eval_at':      [20],
    'n_jobs':            -1,
    'verbose':           -1,
    'random_state':      42,
}


model_honest = lgb.LGBMRanker(**lgbm_params)
model_honest.fit(
    X_train, y_train, group=train_groups,
    eval_set=[(X_val, y_val)], eval_group=[val_groups],
    eval_at=[20], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)
ndcg_honest = scores_to_ndcg(model_honest.predict(X_val), val_rr, val_gt_rr)
print(f"\n✓ HONEST NDCG@20 (train_rr only): {ndcg_honest:.4f}")

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Training until validation scores don't improve for 30 rounds
[50]	valid_0's ndcg@20: 0.525469
[100]	valid_0's ndcg@20: 0.526005
Early stopping, best iteration is:
[81]	valid_0's ndcg@20: 0.526854


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")



✓ HONEST NDCG@20 (train_rr only): 0.1916


In [ ]:
import pandas as pd, numpy as np
cand_check = pd.read_parquet(WORK_DIR / "candidates_val_v3.pq")
n_dup = cand_check.duplicated(['user_id','item_id']).sum()
print(f"Дубликатов (user_id,item_id): {n_dup:,}")

fi = pd.DataFrame({
    'feature': FEAT_COLS,
    'gain': model_honest.booster_.feature_importance(importance_type='gain'),
})
fi['pct'] = fi['gain']/fi['gain'].sum()*100
print(fi.sort_values('gain', ascending=False).head(12).to_string(index=False))


Дубликатов (user_id,item_id): 0
                  feature         gain       pct
                ease_rank 94018.464993 40.105275
    item_popularity_trend 24321.037706 10.374578
              in_hard_neg  9880.592970  4.214745
      item_purchases_2015  9815.784117  4.187100
item_purchases_recent_30d  9588.058331  4.089959
      itemknn_from_last_3  8035.358574  3.427627
                 item_ctr  6944.667058  2.962373
      itemknn_from_last_1  6428.063254  2.742007
               ials_score  5777.290941  2.464408
    n_category_purch_user  5101.573639  2.176168
         itemknn_rank_min  4761.100629  2.030933
  item_purchases_prev_30d  4168.887820  1.778314


In [ ]:
print("Считаем pearson correlation фич с target на train (chunked)...")

corrs = []
n_feat = X_train.shape[1]

y_mean = y_train.mean()
y_centered = y_train - y_mean
y_norm = np.sqrt((y_centered ** 2).sum())

for i in range(n_feat):
    col = X_train[:, i].astype(np.float64)
    c_mean = col.mean()
    c_centered = col - c_mean
    c_norm = np.sqrt((c_centered ** 2).sum())
    if c_norm < 1e-12 or y_norm < 1e-12:
        corr = 0.0
    else:
        corr = float((c_centered * y_centered).sum() / (c_norm * y_norm))
    corrs.append(corr)

corr_df = pd.DataFrame({'feature': FEAT_COLS, 'corr_with_target': corrs})
corr_df['abs_corr'] = corr_df['corr_with_target'].abs()
corr_df = corr_df.sort_values('abs_corr', ascending=False).reset_index(drop=True)

print("\nТоп-15 фич по |corr| с target:")
print(corr_df.head(15)[['feature','corr_with_target']].to_string(index=False))

high_corr = corr_df[corr_df['abs_corr'] > 0.5]
if len(high_corr) > 0:
    print(f"\n⚠ Фичи с |corr| > 0.5 — подозрение на утечку:")
    print(high_corr[['feature','corr_with_target']].to_string(index=False))
else:
    print(f"\n✓ Все фичи с |corr| ≤ 0.5 — утечка маловероятна")
del y_centered, c_centered

Считаем pearson correlation фич с target на train (chunked)...

Топ-15 фич по |corr| с target:
            feature  corr_with_target
          i2i_score          0.090204
          ease_rank         -0.083774
      was_impressed         -0.078555
  itemknn_score_max          0.077573
itemknn_from_last_5          0.077573
itemknn_from_last_3          0.076249
           i2i_rank         -0.074149
  itemknn_score_sum          0.072259
itemknn_from_last_1          0.067347
 ease_rank_x_bucket         -0.061483
      in_i2i_recent          0.059615
   itemknn_rank_min         -0.059218
            in_ease          0.057931
         ials_score          0.054014
  itemknn_n_sources          0.053298

✓ Все фичи с |corr| ≤ 0.5 — утечка маловероятна


In [ ]:
free(
    'X_train', 'y_train', 'train_groups',
    'X_val',   'y_val',   'val_groups',
    'val_rr',  'val_gt_rr',
    'model_honest',
    'fi', 'corr_df', 'corrs', 'y_centered', 'c_centered',
    'X_train_orig', 'X_val_orig', 'model_orig', 'ndcg_orig',
    'train_rr_users', 'val_rr_users', 'train_set', 'val_set',
)
print(f"После очистки: RAM {mem_mb():.0f} MB")

# Final fit на всех кандидатах с Optuna-параметрами (n_estimators=528)
best_lgbm = lgb.LGBMRanker(**lgbm_params)
best_lgbm.fit(X_all, y_all, group=all_groups)
print(f"✓ best_lgbm trained ({lgbm_params['n_estimators']} iters)")

После очистки: RAM 16312 MB


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


✓ best_lgbm trained (528 iters)


In [ ]:
best_lgbm.booster_.save_model(str(WORK_DIR / "lgbm_v2.txt"))
print(f"✓ best_lgbm trained ({528} iters) and saved to lgbm_v2.txt")

save_pickle(FEAT_COLS, "FEAT_COLS_v2")
print(f"FEAT_COLS_v2 saved ({len(FEAT_COLS)} cols)")

free('X_all', 'y_all', 'all_groups')
print(f"RAM перед Section 7: {mem_mb():.0f} MB")

✓ best_lgbm trained (528 iters) and saved to lgbm_v2.txt
FEAT_COLS_v2 saved (71 cols)
RAM перед Section 7: 13624 MB


## 7. Augment test candidates теми же фичами

Загружаем существующий `test_cands.pq` и добавляем ровно те же фичи что в val.


In [ ]:

free('cand', 'X_all', 'y_all', 'all_groups',
     'X_train', 'y_train', 'train_groups',
     'X_val', 'y_val', 'val_groups',
     'val_rr', 'val_gt_rr', 'val_gt_dict',
     'model_honest', 'fi', 'corr_df', 'corrs',
     'B_ease', 'user_item_csr',
     'recent_df_val', 'knn_feat_val',
     'last_authors_val', 'last_series_val', 'last_cats_val',
     'rec_val')

LAST_N = 5
K_I2I_TOTAL = 30  
ITEMKNN_K = 100 


TEST_CANDS_PATH = ART_DIR / "test_cands.pq"
if not TEST_CANDS_PATH.exists():
    TEST_CANDS_PATH = Path("test_cands.pq")
test_cands = pd.read_parquet(TEST_CANDS_PATH)
print(f"test_cands: {test_cands.shape}, RAM: {mem_mb():.0f} MB")


test_users = pd.read_csv(DATA_DIR / "test_users.csv")
test_user_ids = test_users['user_id'].values
print(f"test_user_ids: {len(test_user_ids):,}")

FEAT_COLS_PATH = ART_DIR / "FEAT_COLS_v2.pkl"
if FEAT_COLS_PATH.exists():
    FEAT_COLS = load_pickle("FEAT_COLS_v2")
else:
    META_COLS = ['user_id','item_id','target']
    FEAT_COLS = [c for c in test_cands.columns if c not in META_COLS]
print(f"FEAT_COLS: {len(FEAT_COLS)}")


item_authors_d = load_pickle("item_authors_d")
item_series_d  = load_pickle("item_series_d")
item_cats_d    = load_pickle("item_cats_d")


rec_full          = load_pickle("recency_full")
last_authors_full = load_pickle("last_authors_full")
last_series_full  = load_pickle("last_series_full")
last_cats_full    = load_pickle("last_cats_full")
print(f"recency_full users: {len(rec_full['last_items']):,}")

RECENT_TEST_PATH = ART_DIR / "recent_cands_test.pq"
KNN_TEST_PATH    = ART_DIR / "itemknn_features_test.pq"

if RECENT_TEST_PATH.exists():
    recent_df_test = pd.read_parquet(RECENT_TEST_PATH)
    print(f"recent_df_test: {recent_df_test.shape}")
else:
    recent_df_test = pd.DataFrame(columns=['user_id','item_id','i2i_score','i2i_rank'])
    print("  ⚠ recent_cands_test.pq не найден — i2i от EASE будет пропущен")

if KNN_TEST_PATH.exists():
    knn_feat_test = pd.read_parquet(KNN_TEST_PATH)
    print(f"knn_feat_test: {knn_feat_test.shape}")
else:
    knn_feat_test = pd.DataFrame(columns=[
        'user_id','item_id','itemknn_score_max','itemknn_score_sum',
        'itemknn_rank_min','itemknn_n_sources',
        'itemknn_from_last_1','itemknn_from_last_3','itemknn_from_last_5'
    ])
    print("  ⚠ itemknn_features_test.pq не найден — ItemKNN будет пропущен")


def activity_bucket(n):
    if n == 0: return 0
    if n <= 5: return 1
    if n <= 20: return 2
    if n <= 50: return 3
    return 4


already_bought_full = load_pickle("already_bought_full")
popular_items_full  = load_pickle("popular_items_full")

print(f"\n✓ Section 7 готов. RAM: {mem_mb():.0f} MB")

test_cands: (40764365, 51), RAM: 15136 MB
test_user_ids: 185,282
FEAT_COLS: 72
recency_full users: 303,549
recent_df_test: (5103390, 4)
knn_feat_test: (43649703, 9)

✓ Section 7 готов. RAM: 13547 MB


In [ ]:
# 7a. Merge i2i + ItemKNN фичи

def add_pair_feature(base, src_df, value_col, fill_value, dtype, out_col=None):
    """RAM-friendly merge: добавляет колонку через MultiIndex.map.
    Пик ~1 GB вместо ~6 GB от обычного pd.merge."""
    if out_col is None: out_col = value_col
    src_idx = pd.MultiIndex.from_arrays(
        [src_df['user_id'].to_numpy(), src_df['item_id'].to_numpy()],
        names=['user_id','item_id']
    )
    src_s = pd.Series(src_df[value_col].to_numpy(), index=src_idx)
    base_idx = pd.MultiIndex.from_arrays(
        [base['user_id'].to_numpy(), base['item_id'].to_numpy()],
        names=['user_id','item_id']
    )
    vals = src_s.reindex(base_idx).to_numpy()
    del src_idx, src_s, base_idx
    if np.issubdtype(np.dtype(dtype), np.integer):
        vals = np.nan_to_num(vals, nan=fill_value).astype(dtype)
    else:
        vals = np.nan_to_num(vals, nan=fill_value).astype(dtype)
    base[out_col] = vals
    del vals
    gc.collect()

print(f"test_cands перед 7a: {test_cands.shape}, RAM: {mem_mb():.0f} MB")

# i2i от EASE
print("Adding i2i features...")
add_pair_feature(test_cands, recent_df_test, 'i2i_score', 0.0, np.float32)
add_pair_feature(test_cands, recent_df_test, 'i2i_rank',  K_I2I_TOTAL, np.int32)
test_cands['in_i2i_recent'] = (test_cands['i2i_rank'] < K_I2I_TOTAL).astype(np.int8)
free('recent_df_test')
print(f"  i2i: RAM {mem_mb():.0f} MB")

# ItemKNN (7 фич)
print("Adding ItemKNN features...")
add_pair_feature(test_cands, knn_feat_test, 'itemknn_score_max',    0.0,        np.float32)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_score_sum',    0.0,        np.float32)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_rank_min',     ITEMKNN_K,  np.int32)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_n_sources',    0,          np.int8)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_from_last_1',  0.0,        np.float32)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_from_last_3',  0.0,        np.float32)
add_pair_feature(test_cands, knn_feat_test, 'itemknn_from_last_5',  0.0,        np.float32)
test_cands['in_itemknn'] = (test_cands['itemknn_rank_min'] < ITEMKNN_K).astype(np.int8)
free('knn_feat_test')
print(f"  ItemKNN: RAM {mem_mb():.0f} MB")

print(f"\ntest_cands после 7a: {test_cands.shape}, RAM: {mem_mb():.0f} MB")

test_cands перед 7a: (40764365, 51), RAM: 13547 MB
Adding i2i features...
  i2i: RAM 13599 MB
Adding ItemKNN features...
  ItemKNN: RAM 14659 MB

test_cands после 7a: (40764365, 62), RAM: 14659 MB


In [ ]:
# 7b. Recency-match фичи для test
print("Computing recency-match features for test...")
n = len(test_cands)
lam = np.zeros(n, dtype=np.int8)
lsm = np.zeros(n, dtype=np.int8)
lcm = np.zeros(n, dtype=np.int8)
dslp = np.full(n, -1, dtype=np.float32)
dsli = np.full(n, -1, dtype=np.float32)

_la = last_authors_full; _ls = last_series_full; _lc = last_cats_full
_lpt = rec_full['last_purchase_ts']; _lit = rec_full['last_inter_ts']
ref_ts = rec_full['ref_ts']

uids = test_cands['user_id'].values
iids = test_cands['item_id'].values
CHUNK = 2_000_000

for start in range(0, n, CHUNK):
    end = min(start + CHUNK, n)
    for k in range(start, end):
        uid = int(uids[k]); it = int(iids[k])
        i_a = item_authors_d.get(it, []); i_s = item_series_d.get(it, []); i_c = item_cats_d.get(it, [])
        a_set = _la.get(uid, ()); s_set = _ls.get(uid, ()); c_set = _lc.get(uid, ())
        lam[k] = 1 if any(a in a_set for a in i_a) else 0
        lsm[k] = 1 if any(s in s_set for s in i_s) else 0
        lcm[k] = 1 if any(c in c_set for c in i_c) else 0
        lp = _lpt.get(uid)
        li = _lit.get(uid)
        if lp is not None: dslp[k] = (ref_ts - lp).days
        if li is not None: dsli[k] = (ref_ts - li).days
    print(f"  {end:,}/{n:,}  RAM={mem_mb():.0f} MB")

test_cands['last_author_match'] = lam
test_cands['last_series_match'] = lsm
test_cands['last_category_match'] = lcm
test_cands['days_since_last_purchase'] = dslp
test_cands['days_since_last_interaction'] = dsli
del lam, lsm, lcm, dslp, dsli, uids, iids
free()
print(f"RAM: {mem_mb():.0f} MB")

Computing recency-match features for test...
  2,000,000/40,764,365  RAM=14996 MB
  4,000,000/40,764,365  RAM=15000 MB
  6,000,000/40,764,365  RAM=15007 MB
  8,000,000/40,764,365  RAM=15011 MB
  10,000,000/40,764,365  RAM=15017 MB
  12,000,000/40,764,365  RAM=15023 MB
  14,000,000/40,764,365  RAM=15030 MB
  16,000,000/40,764,365  RAM=15036 MB
  18,000,000/40,764,365  RAM=15042 MB
  20,000,000/40,764,365  RAM=15048 MB
  22,000,000/40,764,365  RAM=15055 MB
  24,000,000/40,764,365  RAM=15061 MB
  26,000,000/40,764,365  RAM=15067 MB
  28,000,000/40,764,365  RAM=15072 MB
  30,000,000/40,764,365  RAM=15078 MB
  32,000,000/40,764,365  RAM=15084 MB
  34,000,000/40,764,365  RAM=15090 MB
  36,000,000/40,764,365  RAM=15097 MB
  38,000,000/40,764,365  RAM=15103 MB
  40,000,000/40,764,365  RAM=15106 MB
  40,764,365/40,764,365  RAM=15108 MB
RAM: 15107 MB


In [ ]:
NEW_TO_OLD = {
    'user_n_interactions':   'n_interactions_x',
    'user_n_purchases':      'n_purchases_x',
    'user_n_unique_bought':  'n_unique_items_bought',
    'user_avg_rating':       'avg_rating_x',
    'user_n_rated':          'n_rated_x',
    'user_purchase_rate':    'purchase_rate_x',
    'item_n_interactions':   'n_interactions_y',
    'item_n_unique_users':   'n_unique_users',
    'item_n_purchases':      'n_purchases_y',
    'item_n_unique_buyers':  'n_unique_buyers',
    'item_avg_rating':       'avg_rating_y',
    'item_n_rated':          'n_rated_y',
    'item_rating_std':       'rating_std',
    'item_n_categories':     'n_categories',
    'item_n_authors':        'n_authors',
    'item_has_series':       'has_series',
    'item_purchase_rate':    'purchase_rate_y',
    'item_popularity_rank':  'popularity_rank',
}
test_cands.rename(columns={k:v for k,v in NEW_TO_OLD.items()
                            if k in test_cands.columns}, inplace=True)

In [ ]:
# 7c. Activity bucket + interactions + impression 
def activity_bucket(n):
    if n <= 0: return 0
    if n <= 5: return 1
    if n <= 20: return 2
    if n <= 50: return 3
    return 4

def _find_act_col(df):
    for c in ['user_n_interactions', 'n_interactions_x']:
        if c in df.columns:
            return c
    return None

_act = _find_act_col(test_cands)
if _act is not None:
    test_cands['user_activity_bucket'] = test_cands[_act].fillna(0).apply(activity_bucket).astype(np.int8)
    print(f"user_activity_bucket из колонки '{_act}'")
else:
    test_cands['user_activity_bucket'] = np.int8(0)
    print("⚠ колонка интеракций не найдена — bucket=0")

bucket_f = test_cands['user_activity_bucket'].astype(np.float32)
for src in ['ease_rank','ials_score','content_rank','toppop_rank','i2i_rank']:
    if src in test_cands.columns:
        test_cands[f'{src}_x_bucket'] = (test_cands[src].astype(np.float32) * bucket_f).astype(np.float32)
del bucket_f

_missing = [c for c in FEAT_COLS if c not in test_cands.columns]
if _missing:
    raise ValueError(
        f"{len(_missing)} фич отсутствуют — НЕ заполняем -1 (это сломает скор!).\n"
        f"Вероятно naming mismatch. Missing: {_missing}"
    )

for col in FEAT_COLS:
    if pd.api.types.is_float_dtype(test_cands[col]):
        test_cands[col] = test_cands[col].fillna(-1).astype(np.float32)
    else:
        test_cands[col] = test_cands[col].fillna(-1)
print(f"test_cands финал: {test_cands.shape}")

user_activity_bucket из колонки 'n_interactions_x'


ValueError: 1 фич отсутствуют — НЕ заполняем -1 (это сломает скор!).
Вероятно naming mismatch. Missing: ['was_impressed']

In [9]:
test_cands.to_parquet(WORK_DIR / "test_cands_v2.pq", index=False)
print(f"test_cands_v2.pq saved: {test_cands.shape}")
print(f"RAM: {mem_mb():.0f} MB")

test_cands_v2.pq saved: (40764365, 73)
RAM: 12444 MB


In [10]:
rows = len(test_cands)

minus1_stats = []

for col in test_cands.columns:
    pct = (test_cands[col].to_numpy() == -1).sum() / rows * 100
    minus1_stats.append((col, pct))

minus1_stats = (
    pd.DataFrame(minus1_stats, columns=["column", "minus1_percent"])
      .sort_values("minus1_percent", ascending=False)
      .reset_index(drop=True)
)

minus1_stats

,column,minus1_percent
0,toppop_rank,54.548047
1,toppop_rank_x_bucket,10.815719
2,days_since_last_purchase,3.721142
3,user_id,0.000000
4,item_id,0.000000
...,...,...
68,days_since_last_interaction,0.000000
69,ease_rank_x_bucket,0.000000
70,ials_score_x_bucket,0.000000
71,content_rank_x_bucket,0.000000


## 8. Predict + submission

In [20]:
import lightgbm as lgb
best_lgbm = lgb.Booster(model_file=str(WORK_DIR / "lgbm_v2.txt"))

In [ ]:
test_cands = pd.read_parquet(ART_DIR /"test_cands_v2.pq")

In [13]:
FEAT_COLS = load_pickle('FEAT_COLS_v2', DIR=WORK_DIR)

In [ ]:
for col in test_cands.columns:
    if col in ('user_id', 'item_id'):
        continue
    dt = test_cands[col].dtype
    if dt == 'float64':
        test_cands[col] = test_cands[col].astype('float32')
    elif dt == 'int64':
        cmin = test_cands[col].min()
        cmax = test_cands[col].max()
        if cmin >= 0 and cmax <= 1:
            test_cands[col] = test_cands[col].astype('int8')      # бинарные флаги
        elif cmin >= -128 and cmax <= 127:
            test_cands[col] = test_cands[col].astype('int8')
        elif cmin >= -2**31 and cmax < 2**31:
            test_cands[col] = test_cands[col].astype('int32')
    gc.collect()
print(f"После downcast: RAM {mem_mb():.0f} MB")
trim()

После downcast: RAM 18890 MB


In [ ]:
# Позиции нужных колонок (учёт возможных дубликатов имён)
cols = list(test_cands.columns)
uid_pos  = cols.index('user_id')
iid_pos  = cols.index('item_id')
ease_pos = cols.index('ease_rank')

ease_small = pd.DataFrame({
    'user_id':   test_cands.iloc[:, uid_pos].to_numpy(),
    'item_id':   test_cands.iloc[:, iid_pos].to_numpy(),
    'ease_rank': test_cands.iloc[:, ease_pos].to_numpy(),
})
print(f"ease_small: {ease_small.shape}, RAM: {mem_mb():.0f} MB")

ease_max = ease_small['ease_rank'].max()  # это fill_value (K_CAND)
ease_real = ease_small[ease_small['ease_rank'] < ease_max]

ease_real = (
    ease_real
    .sort_values(['user_id','ease_rank'], ascending=[True, True])
    .groupby('user_id', sort=False)
    .head(60)
)
ease_per_user = ease_real.groupby('user_id')['item_id'].apply(list).to_dict()
del ease_small, ease_real
gc.collect(); trim()
print(f"EASE fallback: {len(ease_per_user):,} users, RAM: {mem_mb():.0f} MB")


ease_small: (40764365, 3), RAM: 13051 MB
EASE fallback: 170,113 users, RAM: 12974 MB


In [21]:
import pyarrow as pa
import pyarrow.parquet as pq

seen = set()
feat_positions = []
for i, c in enumerate(cols):
    if c in FEAT_COLS and c not in seen:
        feat_positions.append(i)
        seen.add(c)
print(f"Feature positions: {len(feat_positions)}")

PRED_CHUNK = 200_000
n = len(test_cands)
PRED_FILE = "predictions.pq"
writer = None

for start in range(0, n, PRED_CHUNK):
    end = min(start + PRED_CHUNK, n)
    X = test_cands.iloc[start:end, feat_positions].to_numpy(dtype=np.float32, copy=True)
    s = best_lgbm.predict(X).astype(np.float32)
    del X
    chunk_table = pa.table({
        'user_id': test_cands.iloc[start:end, uid_pos].to_numpy(),
        'item_id': test_cands.iloc[start:end, iid_pos].to_numpy(),
        'score':   s,
    })
    del s
    if writer is None:
        writer = pq.ParquetWriter(PRED_FILE, chunk_table.schema, compression='zstd')
    writer.write_table(chunk_table)
    del chunk_table
    gc.collect(); trim()
    print(f"  {end:,}/{n:,}  RAM={mem_mb():.0f} MB")
writer.close()

free('test_cands')
print(f"После free test_cands: RAM {mem_mb():.0f} MB")

result = pd.read_parquet(PRED_FILE)
print(f"result: {result.shape}, RAM: {mem_mb():.0f} MB")


Feature positions: 71
  200,000/40,764,365  RAM=12900 MB
  400,000/40,764,365  RAM=12900 MB
  600,000/40,764,365  RAM=12900 MB
  800,000/40,764,365  RAM=12900 MB
  1,000,000/40,764,365  RAM=12900 MB
  1,200,000/40,764,365  RAM=12900 MB
  1,400,000/40,764,365  RAM=12900 MB
  1,600,000/40,764,365  RAM=12900 MB
  1,800,000/40,764,365  RAM=12900 MB
  2,000,000/40,764,365  RAM=12900 MB
  2,200,000/40,764,365  RAM=12900 MB
  2,400,000/40,764,365  RAM=12900 MB
  2,600,000/40,764,365  RAM=12900 MB
  2,800,000/40,764,365  RAM=12900 MB
  3,000,000/40,764,365  RAM=12900 MB
  3,200,000/40,764,365  RAM=12900 MB
  3,400,000/40,764,365  RAM=12900 MB
  3,600,000/40,764,365  RAM=12900 MB
  3,800,000/40,764,365  RAM=12900 MB
  4,000,000/40,764,365  RAM=12900 MB
  4,200,000/40,764,365  RAM=12900 MB
  4,400,000/40,764,365  RAM=12900 MB
  4,600,000/40,764,365  RAM=12900 MB
  4,800,000/40,764,365  RAM=12900 MB
  5,000,000/40,764,365  RAM=12900 MB
  5,200,000/40,764,365  RAM=12900 MB
  5,400,000/40,764,365  

In [22]:
N_REC = 20

sub_rr = (
    result
    .sort_values(['user_id','score'], ascending=[True, False])
    .groupby('user_id', sort=False)
    .head(N_REC)
    [['user_id','item_id']]
    .reset_index(drop=True)
)
free('result')

counts = sub_rr.groupby('user_id').size()
short = counts[counts < N_REC].index.tolist()
missing = set(test_user_ids) - set(sub_rr['user_id'])
print(f"short: {len(short):,}, missing: {len(missing):,}")

extra = []
# Юзеры с неполным top-20
for uid in short:
    existing = set(sub_rr[sub_rr['user_id'] == uid]['item_id'])
    bought = already_bought_full.get(uid, set())
    needed = N_REC - len(existing)
    fb = []
    # 1) EASE
    for it in ease_per_user.get(uid, []):
        if it not in existing and it not in bought:
            fb.append(it); existing.add(it)
            if len(fb) >= needed: break
    # 2) TopPop добивка
    if len(fb) < needed:
        for it in popular_items_full:
            if it not in existing and it not in bought:
                fb.append(it); existing.add(it)
                if len(fb) >= needed: break
    extra += [{'user_id': uid, 'item_id': it} for it in fb]

# Юзеры вообще без кандидатов
for uid in missing:
    bought = already_bought_full.get(uid, set())
    existing = set(); fb = []
    for it in ease_per_user.get(uid, []):
        if it not in existing and it not in bought:
            fb.append(it); existing.add(it)
            if len(fb) >= N_REC: break
    if len(fb) < N_REC:
        for it in popular_items_full:
            if it not in existing and it not in bought:
                fb.append(it); existing.add(it)
                if len(fb) >= N_REC: break
    extra += [{'user_id': uid, 'item_id': it} for it in fb]

if extra:
    sub_rr = pd.concat([sub_rr, pd.DataFrame(extra)], ignore_index=True)

assert set(sub_rr['user_id']) == set(test_user_ids), "Не все test_users!"
assert (sub_rr.groupby('user_id').size() == N_REC).all(), "Не у всех ровно 20!"

sub_rr.to_csv("submission_improvements.csv", index=False)
print(f"✓ submission: {sub_rr.shape}, users: {sub_rr['user_id'].nunique():,}")


short: 0, missing: 0
✓ submission: (3705640, 2), users: 185,282


In [25]:
items = pd.read_parquet(DATA_DIR/"items.pq")

In [26]:
def validate_submission(sub, test_users, items, topk=20):
    test_user_ids = set(test_users["user_id"].unique())
    sub_user_ids = set(sub["user_id"].unique())
    valid_items = set(items["item_id"].unique())

    print("submission shape:", sub.shape)
    print("test users:", len(test_user_ids))
    print("sub users:", len(sub_user_ids))
    print("unique items in sub:", sub["item_id"].nunique())

    assert sub["user_id"].notna().all(), "NaN in user_id"
    assert sub["item_id"].notna().all(), "NaN in item_id"

    missing_users = test_user_ids - sub_user_ids
    extra_users = sub_user_ids - test_user_ids

    print("missing users:", len(missing_users))
    print("extra users:", len(extra_users))

    assert len(missing_users) == 0, "Some test users are missing"
    assert len(extra_users) == 0, "Submission has users outside test"

    cnt = sub.groupby("user_id")["item_id"].size()
    uniq_cnt = sub.groupby("user_id")["item_id"].nunique()

    print("min rows/user:", cnt.min())
    print("max rows/user:", cnt.max())
    print("min unique items/user:", uniq_cnt.min())
    print("max unique items/user:", uniq_cnt.max())

    assert cnt.min() == topk, "Some users have fewer than topk rows"
    assert cnt.max() == topk, "Some users have more than topk rows"
    assert uniq_cnt.min() == topk, "Some users have duplicate items"

    bad_items = set(sub["item_id"].unique()) - valid_items
    print("bad item ids:", len(bad_items))

    assert len(bad_items) == 0, "Some item_id are not in items"

    dups = sub.duplicated(["user_id", "item_id"]).sum()
    print("duplicate user-item pairs:", dups)

    assert dups == 0, "Duplicate user-item pairs found"

    item_freq = sub["item_id"].value_counts()
    print("top recommended items:")
    print(item_freq.head(10))

    user_lists = (
        sub.groupby("user_id")["item_id"]
           .apply(lambda x: tuple(x.tolist()))
    )
    print("unique recommendation lists:", user_lists.nunique())

    print("OK: submission passed validation")

validate_submission(
    sub=sub_rr,
    test_users=test_users,
    items=items,
    topk=20
)

submission shape: (3705640, 2)
test users: 185282
sub users: 185282
unique items in sub: 10587
missing users: 0
extra users: 0
min rows/user: 20
max rows/user: 20
min unique items/user: 20
max unique items/user: 20
bad item ids: 0
duplicate user-item pairs: 0
top recommended items:
item_id
18467    100065
13628     82680
610       80416
31376     78300
34003     75949
28715     74250
4941      69141
15390     67145
18327     66200
22085     65474
Name: count, dtype: int64
unique recommendation lists: 152457
OK: submission passed validation


In [30]:
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"\nsample_submission shape: {sample_sub.shape}")
print(f"our submission shape   : {sub_rr.shape}")
print(f"Форматы совпадают: {list(sub_rr.columns) == list(sample_sub.columns)}")


sample_submission shape: (3705640, 2)
our submission shape   : (3705640, 2)
Форматы совпадают: True
